# CONDOR–FlexDC Full-Parity Generic Training, Evaluation, Inference Handoff, and Export

This notebook preserves the complete original V3 training workflow while making the model and dataset selection generic. Repository cloning, local/Colab modes, optional Drive paths, direct and multipart uploads, source integration, structural tests, W&B login/resume, balanced training, all checkpoint roles, interruption recovery, validation-only checkpoint selection, one-time untouched test evaluation, profile-holdout extensions, plots, optional post-freeze real FlexDC validation, and complete artifact packaging are all retained. The profile-holdout evaluation is additive; it does not replace the native V3 metrics.

## 0. Environment, repository, upload, W&B, and run controls — EDIT THIS CELL

In [ ]:
from pathlib import Path
import importlib.util, os, platform

IS_COLAB = importlib.util.find_spec("google.colab") is not None
RUN_ENV = "colab" if IS_COLAB else "local_pc"
WORKSPACE = Path("/content/workspace") if IS_COLAB else Path.cwd() / "flexdc_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Repository controls retained from the original notebook.
COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC.git"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"
FORCE_RECLONE = False
UPDATE_EXISTING_REPOS = False
PRESERVE_LOCAL_REPO_CHANGES = True
CLONE_CONDOR_REPO = True
CLONE_FLEXDC_REPO = True
COMDER_ROOT_OVERRIDE = None
FLEXDC_ROOT_OVERRIDE = None
INSTALL_GENERIC_SOURCES_IN_CONDOR = True
INSTALL_FLEXDC_REQUIREMENTS = False
INSTALL_FLEXDC_EDITABLE = False

# Generic code bundle: existing_dir, path_zip, upload_zip.
PACKAGE_MODE = "path_zip" if IS_COLAB else "existing_dir"
PACKAGE_ZIP_INPUT = Path("/content/FlexDC_Generic_Full_Parity_Bundle.zip") if IS_COLAB else None
PACKAGE_DIR_INPUT = None
BUNDLE_ROOT_OVERRIDE = None
AUTO_UPLOAD_PACKAGE = True

# Dataset acquisition: auto, path_zip, existing_csv, upload_zip,
# multipart_upload. The preparation script creates both a full ZIP and
# upload-safe parts with a SHA-256 manifest.
DATASET_MODE = "auto"
DATASET_INPUT = Path("/content/j2_pairwise_profile_holdout_training_bundle.zip") if IS_COLAB else None
DATASET_PART_INPUTS = []
DATASET_UPLOAD_MANIFEST = None
AUTO_UPLOAD_DATASET = True

# Optional and disabled. Direct upload/local paths remain the default.
USE_GOOGLE_DRIVE_DATA = False
DRIVE_DATASET_PATH = ""
DRIVE_PACKAGE_PATH = ""

MODEL_ID = "2x2_train_resnet_gpt2"
# "3x1_llama_holdout"
# "3x1_resnet_holdout"
# "2x2_train_resnet_gpt2"
# "2x2_train_llama_bloom"

RUN_LABEL = "j2_profile_holdout"
TRAINING_SEED = 101
PROFILE_SPLIT_SEED = 20260804
RUN_FINAL_TEST_EVALUATION = True

USE_WANDB = True
WANDB_MODE = "online"       # online, offline, disabled
WANDB_PROJECT = "flexdc-j2-profile-holdout"
WANDB_ENTITY = None
WANDB_RUN_ID_OVERRIDE = None
WANDB_FORCE_RELOGIN = False

DISPLAY_HTML_TABLES = True
WRITE_HTML_FILES = True
AUTO_DOWNLOAD_FINAL_ZIP = True
INCLUDE_FULL_DATASET_IN_ARTIFACT_ZIP = False

# Resume/recovery controls retained from the original notebook.
RESUME_FROM = None
RECOVERY_MODE = False
RECOVERY_UPLOAD_CHECKPOINTS = True
RESTORE_ROLE_AFTER_TRAINING = "best_loss"
SELECTED_CHECKPOINT_ROLE = "best_feasibility"

# Optional post-freeze inference and real simulator validation. This is
# disabled by default only to keep training and final optimization separate;
# when enabled it runs the same complete multi-seed workflow as inference.
RUN_POSTFREEZE_OPTIMIZATION = False
RUN_ACTUAL_FLEXDC_VALIDATION = False
OPTIMIZATION_STARTS = 256
OPTIMIZATION_ITERATIONS = 1000
OPTIMIZATION_TOP_K = 5
OPTIMIZATION_SEEDS = [30, 31, 32]
OPTIMIZATION_VALIDATION_TIMEOUT = 1800
INFERENCE_CASES = []

print("Environment:", RUN_ENV)
print("Platform:", platform.platform())
print("Workspace:", WORKSPACE)
print("Selected model:", MODEL_ID)
print("W&B:", USE_WANDB, WANDB_MODE)


## 1. Install dependencies without replacing Colab’s PyTorch stack

In [ ]:
import importlib, subprocess, sys
required = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib", "tabulate": "tabulate", "tqdm": "tqdm",
    "openpyxl": "openpyxl", "nbformat": "nbformat",
}
if USE_WANDB and WANDB_MODE != "disabled": required["wandb"] = "wandb"
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch, pandas as pd, numpy as np
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 2. Acquire and verify the generic package

In [ ]:
import shutil, zipfile, sys

# Bootstrap only the small amount of logic needed to locate/extract this bundle.
def _direct_upload_one(description):
    if not IS_COLAB:
        raise FileNotFoundError(description)
    from google.colab import files
    print("Upload", description)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"Upload exactly one file; got {list(uploaded)}")
    return (Path("/content") / next(iter(uploaded))).resolve()

if PACKAGE_MODE == "existing_dir":
    PACKAGE_EXTRACT_ROOT = Path(PACKAGE_DIR_INPUT).expanduser().resolve() if PACKAGE_DIR_INPUT else Path.cwd()
elif PACKAGE_MODE in {"path_zip", "upload_zip"}:
    package_candidates = []
    if PACKAGE_ZIP_INPUT is not None:
        package_candidates.append(Path(PACKAGE_ZIP_INPUT).expanduser())
    if IS_COLAB:
        package_candidates += list(Path("/content").glob("*Full_Parity_Bundle*.zip"))
    PACKAGE_ZIP = next((path.resolve() for path in package_candidates if path.exists()), None)
    if PACKAGE_ZIP is None and (PACKAGE_MODE == "upload_zip" or AUTO_UPLOAD_PACKAGE):
        PACKAGE_ZIP = _direct_upload_one("FlexDC_Generic_Full_Parity_Bundle.zip")
    if PACKAGE_ZIP is None:
        raise FileNotFoundError("Generic package ZIP")
    PACKAGE_EXTRACT_ROOT = WORKSPACE / "generic_package"
    if not zipfile.is_zipfile(PACKAGE_ZIP):
        raise zipfile.BadZipFile(PACKAGE_ZIP)
    if PACKAGE_EXTRACT_ROOT.exists():
        shutil.rmtree(PACKAGE_EXTRACT_ROOT)
    PACKAGE_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(PACKAGE_ZIP) as archive:
        bad = archive.testzip()
        if bad:
            raise zipfile.BadZipFile(f"CRC failure: {bad}")
        archive.extractall(PACKAGE_EXTRACT_ROOT)
else:
    raise ValueError(PACKAGE_MODE)

candidate_roots = []
if BUNDLE_ROOT_OVERRIDE is not None:
    candidate_roots.append(Path(BUNDLE_ROOT_OVERRIDE))
candidate_roots += [PACKAGE_EXTRACT_ROOT, Path.cwd(), Path.cwd().parent]
BUNDLE_ROOT = None
for root in candidate_roots:
    root = root.expanduser().resolve()
    if (root / "flexdc_generic_sources").exists():
        BUNDLE_ROOT = root
        break
    if root.exists():
        hits = list(root.rglob("flexdc_generic_sources"))
        if hits:
            BUNDLE_ROOT = hits[0].parent.resolve()
            break
if BUNDLE_ROOT is None:
    raise FileNotFoundError("Could not locate flexdc_generic_sources")
SOURCE_DIR = BUNDLE_ROOT / "flexdc_generic_sources"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from flexdc_colab_orchestration import *
print("Bundle root:", BUNDLE_ROOT)
print("Source directory:", SOURCE_DIR)


## 3. Clone or update CONDOR-FLEXDC and FlexDC

In [ ]:
repo_states = []
COMDER_ROOT = Path(COMDER_ROOT_OVERRIDE).expanduser().resolve() if COMDER_ROOT_OVERRIDE else WORKSPACE / "comder-main"
FLEXDC_ROOT = Path(FLEXDC_ROOT_OVERRIDE).expanduser().resolve() if FLEXDC_ROOT_OVERRIDE else WORKSPACE / "FlexDC"
if CLONE_CONDOR_REPO:
    repo_states.append(clone_or_update_repository(
        name="CONDOR-FLEXDC", url=COMDER_REPO_URL, destination=COMDER_ROOT,
        branch=COMDER_BRANCH, force_reclone=FORCE_RECLONE,
        update_existing=UPDATE_EXISTING_REPOS,
        preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES,
    ))
if CLONE_FLEXDC_REPO:
    repo_states.append(clone_or_update_repository(
        name="FlexDC", url=FLEXDC_REPO_URL, destination=FLEXDC_ROOT,
        branch=FLEXDC_BRANCH, force_reclone=FORCE_RECLONE,
        update_existing=UPDATE_EXISTING_REPOS,
        preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES,
    ))
REPO_MANIFEST = repository_manifest(repo_states, WORKSPACE / "repository_manifest.json")

CONDOR_TRAIN_DIR = COMDER_ROOT / "am_flexdc" / "train"
GENERIC_SOURCE_INSTALL = (
    install_generic_sources_to_condor(SOURCE_DIR, CONDOR_TRAIN_DIR)
    if CLONE_CONDOR_REPO and INSTALL_GENERIC_SOURCES_IN_CONDOR
    else {"files": []}
)

if CLONE_FLEXDC_REPO and INSTALL_FLEXDC_REQUIREMENTS:
    requirements = FLEXDC_ROOT / "requirements.txt"
    if requirements.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
if CLONE_FLEXDC_REPO and INSTALL_FLEXDC_EDITABLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(FLEXDC_ROOT)], check=True)

for state in repo_states:
    print(state.to_dict())
print("Generic sources integrated:", len(GENERIC_SOURCE_INSTALL.get("files", [])))


## 4. Optional Google Drive copy — disabled unless explicitly enabled

In [ ]:
if IS_COLAB and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    drive.mount("/content/drive")
    if DRIVE_PACKAGE_PATH:
        PACKAGE_ZIP_INPUT=Path(DRIVE_PACKAGE_PATH)
    if DRIVE_DATASET_PATH:
        DATASET_INPUT=Path(DRIVE_DATASET_PATH)
    print("Drive paths configured. Rerun Sections 2 and 5 if they changed.")
else:
    print("Google Drive copy skipped. Direct upload/local paths remain active.")

## 5. Acquire, reassemble if necessary, verify, and extract the dataset bundle

In [ ]:
import hashlib, json, re

configured = Path(DATASET_INPUT).expanduser().resolve() if DATASET_INPUT is not None else None
DATASET_SOURCE = None

if DATASET_MODE in {"auto", "path_zip", "existing_csv"} and configured is not None and configured.exists():
    DATASET_SOURCE = configured
elif DATASET_MODE == "multipart_upload" and DATASET_PART_INPUTS:
    parts = [Path(path).expanduser().resolve() for path in DATASET_PART_INPUTS]
    manifest = Path(DATASET_UPLOAD_MANIFEST).expanduser().resolve() if DATASET_UPLOAD_MANIFEST else None
    if manifest:
        DATASET_SOURCE = reassemble_from_manifest(manifest, search_dir=manifest.parent, output_path=WORKSPACE / "reassembled_dataset.zip")
    else:
        DATASET_SOURCE = reassemble_multipart_file(parts, WORKSPACE / "reassembled_dataset.zip")
elif AUTO_UPLOAD_DATASET and IS_COLAB:
    uploaded = upload_files_direct(
        prompt=(
            "Upload the complete profile-holdout dataset ZIP/CSV OR all .part### "
            "files plus the upload_manifest.json produced by prepare_flexdc_training_dataset.py."
        ),
        multiple=True,
    )
    manifests = [path for path in uploaded if path.name.endswith("upload_manifest.json")]
    parts = [path for path in uploaded if re.search(r"\.part\d+$", path.name)]
    if manifests and parts:
        DATASET_SOURCE = reassemble_from_manifest(
            manifests[0], search_dir=manifests[0].parent,
            output_path=WORKSPACE / "reassembled_dataset.zip",
        )
    elif parts:
        DATASET_SOURCE = reassemble_multipart_file(parts, WORKSPACE / "reassembled_dataset.zip")
    else:
        valid = [path for path in uploaded if path.suffix.lower() in {".zip", ".csv"}]
        if len(valid) != 1:
            raise RuntimeError(f"Could not resolve one dataset input from {uploaded}")
        DATASET_SOURCE = valid[0]
else:
    raise FileNotFoundError(f"Dataset input not found: {DATASET_INPUT}")

DATA_EXTRACT_ROOT = WORKSPACE / "dataset"
if DATASET_SOURCE.suffix.lower() == ".zip":
    DATA_ROOT, DATASET_ZIP_INFO = extract_zip_verified(
        DATASET_SOURCE, DATA_EXTRACT_ROOT, clean_destination=True,
        required_fragments=["dataset_manifest.csv", "training_ready.csv"],
        reject_exact_25_mib=True,
    )
else:
    DATA_ROOT = DATASET_SOURCE.parent
    DATASET_ZIP_INFO = {
        "path": str(DATASET_SOURCE), "sha256": sha256_file(DATASET_SOURCE),
        "size_bytes": DATASET_SOURCE.stat().st_size, "entries": 1,
    }
print("Dataset source:", DATASET_SOURCE)
print("Dataset root:", DATA_ROOT)
print("Dataset fingerprint:", DATASET_ZIP_INFO)


## 6. Install the packaged FlexDC runtime files automatically

In [ ]:
RUNTIME_INSTALL={"installed":False,"files":[]}
if FLEXDC_ROOT.exists():
    RUNTIME_INSTALL=install_runtime_bundle(DATA_ROOT,FLEXDC_ROOT)
print("Runtime bundle installed:",RUNTIME_INSTALL.get("installed"))
print("Runtime files:",len(RUNTIME_INSTALL.get("files",[])))
if INSTALL_FLEXDC_EDITABLE:
    requirements=FLEXDC_ROOT/"requirements.txt"
    if requirements.exists(): subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(requirements)],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(FLEXDC_ROOT)],check=True)

## 7. Resolve the selected model dataset and validate its audit

In [ ]:
from flexdc_profile_split import MODEL_SPECS, assign_profile_holdout_split
if MODEL_ID not in MODEL_SPECS: raise KeyError(f"Unknown MODEL_ID={MODEL_ID}; choices={list(MODEL_SPECS)}")
MODEL_SPEC=MODEL_SPECS[MODEL_ID]

def safe_tag(value): return re.sub(r"[^A-Za-z0-9._-]+","_",str(value)).strip("_") or "run"
csv_candidates=sorted(DATA_ROOT.rglob("*.csv")) if DATA_ROOT.is_dir() else [DATA_ROOT]
model_specific=[p for p in csv_candidates if MODEL_ID.lower() in p.name.lower() and "training_ready" in p.name.lower()]
if model_specific:
    RESULTS_CSV=max(model_specific,key=lambda p:p.stat().st_size)
else:
    master=[p for p in csv_candidates if "master_training_ready" in p.name.lower()]
    if not master: raise FileNotFoundError("No model-specific or master training-ready CSV")
    frame=pd.read_csv(max(master,key=lambda p:p.stat().st_size),low_memory=False)
    frame,audit=assign_profile_holdout_split(frame,MODEL_ID,split_seed=PROFILE_SPLIT_SEED,expected_jobs=2,strict_master_coverage=True)
    generated=WORKSPACE/"generated_splits"; generated.mkdir(exist_ok=True)
    RESULTS_CSV=generated/f"{MODEL_ID}_training_ready.csv"; frame.to_csv(RESULTS_CSV,index=False)
    write_json(generated/f"{MODEL_ID}_audit.json",audit)

audit_candidates=[p for p in DATA_ROOT.rglob("*.json") if MODEL_ID.lower() in p.name.lower() and "audit" in p.name.lower()]
DATASET_AUDIT=json.loads(audit_candidates[0].read_text()) if audit_candidates else None
if DATASET_AUDIT is not None and DATASET_AUDIT.get("status")!="PASS": raise RuntimeError(DATASET_AUDIT)

RUN_NAME=safe_tag(f"{RUN_LABEL}_{MODEL_ID}_seed{TRAINING_SEED}")
MODELS_DIR=WORKSPACE/"models"/RUN_NAME
RESULTS_DIR=WORKSPACE/"results"/RUN_NAME
PLOTS_DIR=RESULTS_DIR/"plots"
for p in [MODELS_DIR,RESULTS_DIR,PLOTS_DIR]: p.mkdir(parents=True,exist_ok=True)
print("Model:",MODEL_SPEC.display_name)
print("Training-ready CSV:",RESULTS_CSV)
print("Rows:",sum(1 for _ in open(RESULTS_CSV,encoding="utf-8",errors="ignore"))-1)
print("Run name:",RUN_NAME)

## 8. Compile generic and original repository sources; run structural tests

In [ ]:
import os, subprocess
compile_targets=sorted(SOURCE_DIR.glob("*.py"))
repo_train=COMDER_ROOT/"am_flexdc"/"train"
for name in ["data_center_model_flexdc_behavior_v3.py","am_flexdc_behavior_training_utilities_v3.py","test_flexdc_behavior_training_v3.py"]:
    path=repo_train/name
    if path.exists(): compile_targets.append(path)
for path in compile_targets: subprocess.run([sys.executable,"-m","py_compile",str(path)],check=True)
env=os.environ.copy(); env["PYTHONPATH"]=str(SOURCE_DIR)+os.pathsep+env.get("PYTHONPATH","")
for test in [SOURCE_DIR/"test_flexdc_behavior_training.py",SOURCE_DIR/"test_flexdc_profile_holdout.py"]:
    subprocess.run([sys.executable,str(test)],cwd=SOURCE_DIR,env=env,check=True)
repo_test=repo_train/"test_flexdc_behavior_training_v3.py"
if repo_test.exists(): subprocess.run([sys.executable,str(repo_test)],cwd=repo_train,check=True)
print("Compilation and structural tests: PASS")

## 9. Import model, training, evaluation, inference, and presentation utilities

In [ ]:
import json, random, numpy as np, pandas as pd, torch
from IPython.display import HTML, display
from data_center_model_flexdc_behavior import DataCenterBehaviorModel, FlexDCBehaviorModelConfig
from flexdc_behavior_training_utilities import (
    FlexDCBehaviorConstants, choose_device, context_metrics_table,
    evaluate_behavior_loader, final_behavior_evaluation, final_behavior_test_evaluation,
    load_behavior_model_checkpoint, prepare_behavior_data, sample_prediction_rows,
    train_behavior_model,
)
from flexdc_behavior_evaluation import (
    log_evaluation_to_wandb, render_styled_table, run_profile_holdout_evaluation,
    write_evaluation_outputs,
)
from flexdc_behavior_inference_utilities import (
    OptimizationSettings, dataframe_for_csv, load_behavior_model,
)
from flexdc_inference_orchestration import (
    ScenarioDefinition, run_scenario_suite, safe_tag,
)
from flexdc_presentation import build_report, write_report

def show_table(frame, caption=None, precision=5, max_rows=500):
    if frame is None or len(frame) == 0:
        print((caption or "Table") + ": no rows")
        return
    sample = frame.head(max_rows).copy()
    if DISPLAY_HTML_TABLES:
        render_styled_table(sample, caption=caption, precision=precision)
    else:
        print((caption or "") + "\n" + sample.to_string(index=False))
print("Imports: PASS")


## 10. W&B login

In [ ]:
wandb_module = None
if USE_WANDB and WANDB_MODE != "disabled":
    import wandb, getpass
    wandb_module = wandb
    os.environ["WANDB_MODE"] = WANDB_MODE
    if WANDB_MODE == "online":
        if WANDB_FORCE_RELOGIN:
            wandb.login(key=getpass.getpass("Paste W&B API key: "), relogin=True)
        else:
            try:
                wandb.login()
            except Exception:
                wandb.login(key=getpass.getpass("Paste W&B API key: "), relogin=True)
    print("W&B authenticated")
else:
    print("W&B disabled")


## 11. Full training configuration — EDIT HERE

In [ ]:
random.seed(TRAINING_SEED); np.random.seed(TRAINING_SEED); torch.manual_seed(TRAINING_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(TRAINING_SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE=2048
NUM_WORKERS=0
REPEAT_GROUP_NORMALIZATION=True
SAMPLER_MODE="balanced"
SAMPLER_NATURAL_FRACTION=0.50
SAMPLER_CONTEXT_FRACTION=0.25
SAMPLER_PRIORITY_FRACTION=0.25
SAMPLER_FEASIBLE_BOOST=4.0
SAMPLER_TRACKING_BOUNDARY_BOOST=2.0
SAMPLER_QOS_BOUNDARY_BOOST=3.0
MAX_EPOCHS=200
BASE_LR=3e-4
MIN_LR=1e-6
WARMUP_EPOCHS=5
WEIGHT_DECAY=1e-4
GRADIENT_CLIP_NORM=1.0
EARLY_STOPPING_PATIENCE=30
EARLY_STOPPING_MIN_DELTA=1e-5
MEAN_TRACKING_LOSS_WEIGHT=1.0
P90_TRACKING_LOSS_WEIGHT=1.0
QOS_LOSS_WEIGHT=4.0
TRACKING_BOUNDARY_MULTIPLIER=2.0
QOS_BOUNDARY_MULTIPLIER=2.0
MODEL_CONFIG=FlexDCBehaviorModelConfig(
    dim_job_mix=13,dim_dc_features=12,st_dim_hidden=512,st_num_heads=4,
    global_projection_dim=128,linear_dim_hidden=512,qos_projection_dim=256,
    skip_connections=True,layer_norm=True,include_masked_mean_pool=True,
)
CONSTANTS=FlexDCBehaviorConstants()
print("Device:",DEVICE)
print("Model config:",MODEL_CONFIG.to_dict())

## 12. Prepare the preassigned split, normalize on training only, and audit seed groups

In [ ]:
behavior_data=prepare_behavior_data(
    results_csv=RESULTS_CSV,diagnostics_csv=None,batch_size=BATCH_SIZE,
    split_seed=PROFILE_SPLIT_SEED,split_mode="preassigned",split_column="Data_Split",
    num_workers=NUM_WORKERS,deduplicate=False,constants=CONSTANTS,
    sampler_mode=SAMPLER_MODE,sampler_seed=TRAINING_SEED,
    natural_fraction=SAMPLER_NATURAL_FRACTION,context_fraction=SAMPLER_CONTEXT_FRACTION,
    priority_fraction=SAMPLER_PRIORITY_FRACTION,feasible_boost=SAMPLER_FEASIBLE_BOOST,
    tracking_boundary_boost=SAMPLER_TRACKING_BOUNDARY_BOOST,
    qos_boundary_boost=SAMPLER_QOS_BOUNDARY_BOOST,
    repeat_group_normalization=REPEAT_GROUP_NORMALIZATION,
)
audit_table=pd.DataFrame([
 {"Item":"Training rows","Value":behavior_data.metadata.train_row_count},
 {"Item":"Validation rows","Value":behavior_data.metadata.validation_row_count},
 {"Item":"Test rows","Value":behavior_data.metadata.test_row_count},
 {"Item":"Training base groups","Value":behavior_data.metadata.train_group_count},
 {"Item":"Validation base groups","Value":behavior_data.metadata.validation_group_count},
 {"Item":"Test base groups","Value":behavior_data.metadata.test_group_count},
 {"Item":"Group overlap","Value":behavior_data.audit["group_overlap"]},
 {"Item":"Repeated base groups","Value":behavior_data.audit["repeated_base_groups"]},
 {"Item":"Training feasible rows","Value":behavior_data.audit["train_actual_feasible_rows"]},
 {"Item":"Validation feasible rows","Value":behavior_data.audit["validation_actual_feasible_rows"]},
 {"Item":"Test feasible rows","Value":behavior_data.audit["test_actual_feasible_rows"]},
])
show_table(audit_table,"Dataset and seed-group audit",0)
show_table(pd.DataFrame([{"Setting":k,"Value":v} for k,v in behavior_data.audit.get("sampler",{}).items()]),"Balanced sampler",6)
context_split=pd.DataFrame(behavior_data.audit.get("context_split_summary",[]))
show_table(context_split,"Per-context split",0)
print("Token features:",behavior_data.metadata.token_feature_names)
print("Global features:",behavior_data.metadata.global_feature_names)
print("The test loader is not evaluated until the checkpoint is frozen.")

## 13. Create the masked Set Transformer and run a real-batch smoke test

In [ ]:
model=DataCenterBehaviorModel(MODEL_CONFIG)
print(model)
print(f"Parameters: {model.parameter_count():,}")
batch=next(iter(behavior_data.train_loader))
with torch.no_grad(): output=model(batch["features"],batch["workload"],batch["mask"])
print("Global input:",tuple(batch["features"].shape))
print("Token input:",tuple(batch["workload"].shape))
print("Mask:",tuple(batch["mask"].shape))
print("Tracking output:",tuple(output["tracking_logs"].shape))
print("QoS output:",tuple(output["qos_probabilities"].shape))

## 14. Start or resume the W&B run

In [ ]:
run=None
training_config={
 "model_id":MODEL_ID,"model_display_name":MODEL_SPEC.display_name,"split_type":MODEL_SPEC.split_type,
 "visible_profiles":MODEL_SPEC.visible_profiles,"heldout_profiles":MODEL_SPEC.heldout_profiles,
 "training_seed":TRAINING_SEED,"profile_split_seed":PROFILE_SPLIT_SEED,
 "dataset":str(RESULTS_CSV),"dataset_sha256":sha256_file(RESULTS_CSV),
 "train_rows":behavior_data.metadata.train_row_count,"validation_rows":behavior_data.metadata.validation_row_count,
 "test_rows":behavior_data.metadata.test_row_count,"batch_size":BATCH_SIZE,"max_epochs":MAX_EPOCHS,
 "base_lr":BASE_LR,"min_lr":MIN_LR,"warmup_epochs":WARMUP_EPOCHS,
 "weight_decay":WEIGHT_DECAY,"gradient_clip_norm":GRADIENT_CLIP_NORM,
 "early_stopping_patience":EARLY_STOPPING_PATIENCE,"qos_loss_weight":QOS_LOSS_WEIGHT,
 "sampler_mode":SAMPLER_MODE,"model_config":MODEL_CONFIG.to_dict(),"device":DEVICE,
 "repository_states":[state.to_dict() for state in repo_states],
}
if wandb_module is not None:
    resume_id=WANDB_RUN_ID_OVERRIDE
    if resume_id is None and RESUME_FROM and Path(RESUME_FROM).exists():
        resume_id=torch.load(RESUME_FROM,map_location="cpu",weights_only=False).get("wandb_run_id")
    run=wandb_module.init(
        project=WANDB_PROJECT,entity=WANDB_ENTITY,name=RUN_NAME,group=MODEL_ID,
        mode=WANDB_MODE,config=training_config,id=resume_id,
        resume="must" if resume_id else None,
    )
    print("W&B run:",run.url if getattr(run,"url",None) else run.id)
else: print("W&B disabled")

## 15. Train with scheduling, clipping, early stopping, complete checkpoints, and resume support

In [ ]:
training_result_model=None
training_result=None
if not RECOVERY_MODE:
    training_result_model,training_result=train_behavior_model(
        model,behavior_data,epochs=MAX_EPOCHS,base_lr=BASE_LR,min_lr=MIN_LR,
        warmup_epochs=WARMUP_EPOCHS,weight_decay=WEIGHT_DECAY,
        gradient_clip_norm=GRADIENT_CLIP_NORM,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,device_name=DEVICE,
        mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
        p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,qos_weight=QOS_LOSS_WEIGHT,
        tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
        qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
        checkpoint_dir=MODELS_DIR,checkpoint_prefix=RUN_NAME,
        model_config=MODEL_CONFIG.to_dict(),training_config=training_config,
        resume_from=RESUME_FROM,restore_role=RESTORE_ROLE_AFTER_TRAINING,
        wandb_run=run,metrics_every_n_epochs=1,verbose=True,
    )
    model=training_result_model
    history=training_result.history
    HISTORY_CSV=RESULTS_DIR/f"{RUN_NAME}_history.csv"
    history.to_csv(HISTORY_CSV,index=False)
    show_table(pd.DataFrame([{"Role":r,"Path":p,"Exists":Path(p).exists()} for r,p in training_result.checkpoint_paths.items()]),"Saved checkpoints")
    print(json.dumps(training_result.summary,indent=2))
else:
    print("RECOVERY_MODE=True: skip training and run the recovery section next.")

## 16. Interruption/checkpoint recovery — use after a runtime interruption

In [ ]:
from flexdc_behavior_training_utilities import TrainingResult
if RECOVERY_MODE:
    if RECOVERY_UPLOAD_CHECKPOINTS and IS_COLAB:
        uploaded=upload_files_direct(prompt="Upload the saved latest/best/final .pt checkpoints and history CSV if available.",multiple=True)
        for source in uploaded:
            if source.suffix.lower()==".pt": shutil.copy2(source,MODELS_DIR/source.name)
            elif source.suffix.lower()==".csv" and "history" in source.name.lower(): shutil.copy2(source,RESULTS_DIR/source.name)
    roles=["latest","best_loss","best_objective","best_feasibility","final"]
    checkpoint_paths={}
    for role in roles:
        matches=sorted(MODELS_DIR.glob(f"*{role}*.pt"))
        if matches: checkpoint_paths[role]=str(matches[-1])
    if "latest" not in checkpoint_paths: raise FileNotFoundError("Recovery requires a latest checkpoint")
    latest=torch.load(checkpoint_paths["latest"],map_location="cpu",weights_only=False)
    if "final" not in checkpoint_paths:
        final=MODELS_DIR/f"{RUN_NAME}_final.pt"; shutil.copy2(checkpoint_paths["latest"],final); checkpoint_paths["final"]=str(final)
    history=pd.DataFrame(latest.get("history",[]))
    if history.empty: raise RuntimeError("Latest checkpoint lacks history")
    HISTORY_CSV=RESULTS_DIR/f"{RUN_NAME}_history.csv"; history.to_csv(HISTORY_CSV,index=False)
    training_result=TrainingResult(
        history=history,
        summary={"recovered":True,"final_epoch":int(latest.get("epoch",-1)),"best_scores":latest.get("best_scores",{}),"best_epochs":latest.get("best_epochs",{})},
        checkpoint_paths=checkpoint_paths,
    )
    show_table(pd.DataFrame([{"Role":r,"Path":p} for r,p in checkpoint_paths.items()]),"Recovered checkpoints")
else:
    print("Recovery skipped.")

## 17. Compare every checkpoint on validation only

In [ ]:
device=choose_device(DEVICE); comparison=[]
for role in ["best_loss","best_objective","best_feasibility","final"]:
    path_text=training_result.checkpoint_paths.get(role) if training_result else None
    if not path_text or not Path(path_text).exists(): continue
    candidate,checkpoint=load_behavior_model_checkpoint(path_text,model_class=DataCenterBehaviorModel,config_class=FlexDCBehaviorModelConfig,device_name=DEVICE)
    vm,_=evaluate_behavior_loader(
        candidate,behavior_data.validation_loader,device=device,metadata=behavior_data.metadata,prefix="validation",
        mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,
        qos_weight=QOS_LOSS_WEIGHT,tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,
        qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,return_rows=False,include_workload_metrics=True,
    )
    comparison.append({
      "Role":role,"Epoch":checkpoint.get("epoch"),"Validation Loss":vm["validation/loss/total"],
      "Objective R2":vm["validation/cost/full_objective/r2"],"Objective Spearman":vm["validation/cost/full_objective/spearman"],
      "P90 R2":vm["validation/tracking/p90_physical/r2"],"Per-job Pj MAE":vm["validation/qos/per_job_probability/mae"],
      "Max-Pj MAE":vm["validation/qos/max_probability/mae"],
      "Feasible Precision":vm["validation/feasibility/combined/feasible_precision"],
      "Feasible Recall":vm["validation/feasibility/combined/actual_feasible_accuracy"],
      "Feasible F1":vm["validation/feasibility/combined/f1_feasible"],
      "False-Feasible Rate":vm["validation/feasibility/combined/false_feasible_rate"],
    })
checkpoint_comparison=pd.DataFrame(comparison)
CHECKPOINT_COMPARISON_CSV=RESULTS_DIR/f"{RUN_NAME}_checkpoint_comparison.csv"
checkpoint_comparison.to_csv(CHECKPOINT_COMPARISON_CSV,index=False)
show_table(checkpoint_comparison,"Validation-only checkpoint comparison")
if run is not None: run.log({"validation_checkpoint_comparison":wandb_module.Table(dataframe=checkpoint_comparison)})
print("The test split has not been used for checkpoint selection.")

## 18. Freeze the selected checkpoint and evaluate train + validation

In [ ]:
selected_path=Path(training_result.checkpoint_paths.get(SELECTED_CHECKPOINT_ROLE,""))
if not selected_path.exists():
    print(f"{SELECTED_CHECKPOINT_ROLE} unavailable; falling back to best_loss")
    SELECTED_CHECKPOINT_ROLE="best_loss"; selected_path=Path(training_result.checkpoint_paths[SELECTED_CHECKPOINT_ROLE])
model,selected_checkpoint=load_behavior_model_checkpoint(
    selected_path,model_class=DataCenterBehaviorModel,config_class=FlexDCBehaviorModelConfig,device_name=DEVICE,
)
metrics,train_predictions,validation_predictions=final_behavior_evaluation(
    model,behavior_data,device_name=DEVICE,mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
    p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,qos_weight=QOS_LOSS_WEIGHT,
    tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
)
validation_context=context_metrics_table(validation_predictions)
METRICS_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_train_validation_metrics.csv"
TRAIN_PREDICTIONS_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_train_predictions.csv"
VALIDATION_PREDICTIONS_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_validation_predictions.csv"
VALIDATION_CONTEXT_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_validation_context.csv"
pd.DataFrame([metrics]).to_csv(METRICS_CSV,index=False)
train_predictions.to_csv(TRAIN_PREDICTIONS_CSV,index=False)
validation_predictions.to_csv(VALIDATION_PREDICTIONS_CSV,index=False)
validation_context.to_csv(VALIDATION_CONTEXT_CSV,index=False)
show_table(pd.DataFrame([{"Metric":k,"Value":v} for k,v in metrics.items() if k.startswith("validation/") and isinstance(v,(int,float,np.number))]),"Locked checkpoint validation metrics",5)
show_table(validation_context,"Validation by workload/context",5)
print("Checkpoint frozen:",selected_path)

## 19. One-time untouched native test evaluation

In [ ]:
test_metrics={}; test_predictions=pd.DataFrame(); test_context=pd.DataFrame()
TEST_METRICS_CSV=TEST_PREDICTIONS_CSV=TEST_CONTEXT_CSV=None
if RUN_FINAL_TEST_EVALUATION:
    test_metrics,test_predictions=final_behavior_test_evaluation(
        model,behavior_data,device_name=DEVICE,mean_tracking_weight=MEAN_TRACKING_LOSS_WEIGHT,
        p90_tracking_weight=P90_TRACKING_LOSS_WEIGHT,qos_weight=QOS_LOSS_WEIGHT,
        tracking_boundary_multiplier=TRACKING_BOUNDARY_MULTIPLIER,qos_boundary_multiplier=QOS_BOUNDARY_MULTIPLIER,
    )
    test_context=context_metrics_table(test_predictions)
    TEST_METRICS_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_metrics.csv"
    TEST_PREDICTIONS_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_predictions.csv"
    TEST_CONTEXT_CSV=RESULTS_DIR/f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_test_context.csv"
    pd.DataFrame([test_metrics]).to_csv(TEST_METRICS_CSV,index=False)
    test_predictions.to_csv(TEST_PREDICTIONS_CSV,index=False)
    test_context.to_csv(TEST_CONTEXT_CSV,index=False)
    show_table(pd.DataFrame([{"Metric":k,"Value":v} for k,v in test_metrics.items() if isinstance(v,(int,float,np.number))]),"Untouched native test metrics",5)
    show_table(test_context,"Untouched test by workload/context",5)
else: print("Final test evaluation disabled.")

## 20. Profile-holdout, boundary, failure, constrained-ranking, and seed-stability evaluation

In [ ]:
PROFILE_EVALUATION_FILES=[]
PROFILE_EVALUATION={}
if RUN_FINAL_TEST_EVALUATION:
    source_test=behavior_data.test_dataframe.reset_index(drop=True)
    PROFILE_EVALUATION=run_profile_holdout_evaluation(test_predictions,source_test)
    PROFILE_EVALUATION_FILES=write_evaluation_outputs(
        PROFILE_EVALUATION,RESULTS_DIR,prefix=f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}",write_html=WRITE_HTML_FILES,
    )
    for key in ["overall_metrics","metrics_by_benchmark","metrics_by_benchmark_workload_type","boundary_metrics","failure_type_confusion","ranking_metrics","seed_stability"]:
        show_table(PROFILE_EVALUATION.get(key),key.replace("_"," ").title(),5)
    if run is not None: log_evaluation_to_wandb(run,PROFILE_EVALUATION,namespace="profile_holdout")
    print("Evaluation outputs:",len(PROFILE_EVALUATION_FILES))

## 21. Send selected metrics, tables, and samples to W&B

In [ ]:
if run is not None:
    for namespace,payload in [("selected",metrics),("final",test_metrics)]:
        for key,value in payload.items():
            if isinstance(value,(int,float,np.integer,np.floating)) and np.isfinite(value): run.summary[f"{namespace}/{key}"]=float(value)
    run.summary["selected_checkpoint_role"]=SELECTED_CHECKPOINT_ROLE
    run.summary["selected_checkpoint_epoch"]=int(selected_checkpoint.get("epoch",-1))
    run.summary["selected_checkpoint_path"]=str(selected_path)
    run.summary["test_evaluated_after_checkpoint_lock"]=bool(RUN_FINAL_TEST_EVALUATION)
    payload={
      "validation_context_summary":wandb_module.Table(dataframe=validation_context),
      "validation_prediction_sample":wandb_module.Table(dataframe=sample_prediction_rows(validation_predictions,max_rows=2048,seed=0)),
    }
    if RUN_FINAL_TEST_EVALUATION:
        payload.update({"final_test_context_summary":wandb_module.Table(dataframe=test_context),"final_test_prediction_sample":wandb_module.Table(dataframe=sample_prediction_rows(test_predictions,max_rows=2048,seed=0))})
    run.log(payload)
    print("W&B summaries updated")
else: print("W&B disabled")

## 22. Plot and save all training diagnostics

In [ ]:
import matplotlib.pyplot as plt
curve_specs=[
 ("train/loss/total","Training weighted loss"),("validation/loss/total","Validation weighted loss"),
 ("learning_rate","Learning rate"),("validation/cost/full_objective/r2","Validation objective R²"),
 ("validation/feasibility/combined/feasible_precision","Validation feasible precision"),
 ("validation/feasibility/combined/actual_feasible_accuracy","Validation feasible recall"),
 ("validation/qos/max_probability/mae","Validation max-Pj MAE"),
 ("validation/tracking/p90_physical/r2","Validation p90 R²"),
]
PLOT_FILES=[]
for column,title in curve_specs:
    if column not in history.columns: continue
    fig=plt.figure(figsize=(8,4)); plt.plot(history["epoch"],history[column]); plt.xlabel("Epoch"); plt.ylabel(title); plt.title(title); plt.grid(True,alpha=.3); plt.tight_layout()
    path=PLOTS_DIR/f"{safe_tag(column)}.png"; fig.savefig(path,dpi=160,bbox_inches="tight"); PLOT_FILES.append(path); plt.show(); plt.close(fig)
print("Saved plots:",len(PLOT_FILES))

## 23. Optional post-freeze optimization and real FlexDC validation

In [ ]:
OPTIMIZATION_OUTPUT_FILES = []
POSTFREEZE_RESULTS = []
POSTFREEZE_SUMMARY = pd.DataFrame()
if RUN_POSTFREEZE_OPTIMIZATION:
    if not INFERENCE_CASES:
        print("No INFERENCE_CASES supplied; deriving one seen, one one-held, and one two-held workload from the test dataset.")
        catalog = behavior_data.test_dataframe[
            ["Workload_Name", "Benchmark_Type", "workload_config", "experiment_config", "server_count", "utilization"]
        ].drop_duplicates()
        for benchmark in ["Seen-Profile Baseline", "One Held-Out Profile", "Two Held-Out Profiles"]:
            subset = catalog[catalog["Benchmark_Type"].astype(str) == benchmark]
            if len(subset):
                row = subset.iloc[0]
                INFERENCE_CASES.append({
                    "case_id": safe_tag(f"{benchmark}_{row['Workload_Name']}_u{row['utilization']}"),
                    **row.to_dict(),
                })

    loaded = load_behavior_model(selected_path, device_name=DEVICE)
    runtime = {"label": RUN_NAME, "loaded": loaded, "model_id": MODEL_ID, "checkpoint": selected_path}
    cases = []
    for case in INFERENCE_CASES:
        workload_value = case.get("workload_config") or case.get("workload")
        experiment_value = case.get("experiment_config") or case.get("experiment")
        workload_path = Path(workload_value)
        experiment_path = Path(experiment_value)
        if not workload_path.is_absolute():
            workload_path = FLEXDC_ROOT / workload_path
        if not experiment_path.is_absolute():
            experiment_path = FLEXDC_ROOT / experiment_path
        cases.append(ScenarioDefinition(
            case_id=case["case_id"], workload_config=str(workload_path),
            experiment_config=str(experiment_path),
            server_count=int(case.get("server_count", 1000)),
            utilization=float(case.get("utilization", 0.8)),
            initial_pbar=case.get("initial_pbar"), initial_r=case.get("initial_r"),
            initial_weights=tuple(case["initial_weights"]) if case.get("initial_weights") else None,
            category=case.get("Benchmark_Type"),
        ))

    postfreeze_settings = OptimizationSettings(
        starts=OPTIMIZATION_STARTS, iterations=OPTIMIZATION_ITERATIONS,
        top_k=OPTIMIZATION_TOP_K, random_seed=TRAINING_SEED,
        r_over_p_max=0.6, weight_min=0.3, weight_max=0.7,
    )
    POSTFREEZE_RESULTS, POSTFREEZE_SUMMARY = run_scenario_suite(
        model_runtimes=[runtime], cases=cases,
        output_root=RESULTS_DIR / "postfreeze_optimization",
        settings=postfreeze_settings,
        tracking_margin=0.04, qos_margin=0.01,
        run_flexdc=RUN_ACTUAL_FLEXDC_VALIDATION,
        validate_anchor=True, validate_top_k=True,
        simulator_seeds=OPTIMIZATION_SEEDS,
        flexdc_root=FLEXDC_ROOT,
        gradient_config=FLEXDC_ROOT / "configs/gradient_descent/gradient_descent_j2_pairwise_rsr.ini",
        cluster_config=FLEXDC_ROOT / "configs/cluster/cluster.ini",
        validation_timeout_seconds=OPTIMIZATION_VALIDATION_TIMEOUT,
        resume=True, dry_run_flexdc=False,
    )
    show_table(POSTFREEZE_SUMMARY, "Post-freeze optimization and simulator-validation summary", 5)
    for result in POSTFREEZE_RESULTS:
        result_dir = Path(result.output_dir)
        OPTIMIZATION_OUTPUT_FILES.extend(path for path in result_dir.rglob("*") if path.is_file())
else:
    print("Post-freeze optimization disabled. The full inference notebook contains the same workflow with batch and comparison controls.")


## 24. Package complete artifacts, log the W&B artifact, and optionally download

In [ ]:
RUN_CONFIG_JSON = write_json(RESULTS_DIR / f"{RUN_NAME}_run_config.json", {
    "training_config": training_config,
    "selected_checkpoint_role": SELECTED_CHECKPOINT_ROLE,
    "selected_checkpoint": str(selected_path),
    "dataset_source": str(DATASET_SOURCE),
    "dataset_sha256": DATASET_ZIP_INFO.get("sha256"),
    "runtime_install": RUNTIME_INSTALL,
    "generic_source_install": GENERIC_SOURCE_INSTALL,
    "repositories": [state.to_dict() for state in repo_states],
    "postfreeze_summary_rows": int(len(POSTFREEZE_SUMMARY)),
})

# Produce a compact human-readable report in addition to CSV/W&B outputs.
summary_tables = {
    "Checkpoint comparison": checkpoint_comparison,
    "Validation contexts": validation_context,
    "Untouched test contexts": test_context,
    "Profile benchmarks": PROFILE_EVALUATION.get("metrics_by_benchmark", pd.DataFrame()) if PROFILE_EVALUATION else pd.DataFrame(),
    "Boundary metrics": PROFILE_EVALUATION.get("boundary_metrics", pd.DataFrame()) if PROFILE_EVALUATION else pd.DataFrame(),
    "Post-freeze optimization": POSTFREEZE_SUMMARY,
}
TRAINING_REPORT_HTML = write_report(
    RESULTS_DIR / f"{RUN_NAME}_presentation_ready_report.html",
    build_report(
        title="CONDOR–FlexDC generic training and profile-holdout evaluation",
        subtitle=f"{MODEL_SPEC.display_name}; selected checkpoint {SELECTED_CHECKPOINT_ROLE}",
        tables={name: frame for name, frame in summary_tables.items() if frame is not None and len(frame)},
        notes=[
            "Checkpoint selection used validation only.",
            "The profile-holdout tables are additive to the native V3 train/validation/test metrics.",
        ],
    ),
)

artifact_paths = [
    HISTORY_CSV, CHECKPOINT_COMPARISON_CSV, METRICS_CSV,
    TRAIN_PREDICTIONS_CSV, VALIDATION_PREDICTIONS_CSV,
    VALIDATION_CONTEXT_CSV, selected_path, RUN_CONFIG_JSON,
    REPO_MANIFEST, SOURCE_DIR, PLOTS_DIR, TRAINING_REPORT_HTML,
    *PROFILE_EVALUATION_FILES, *OPTIMIZATION_OUTPUT_FILES,
]
for optional in [TEST_METRICS_CSV, TEST_PREDICTIONS_CSV, TEST_CONTEXT_CSV]:
    if optional is not None:
        artifact_paths.append(optional)
for value in training_result.checkpoint_paths.values():
    if Path(value).exists():
        artifact_paths.append(Path(value))

# Preserve locally generated workloads, experiment/gradient/cluster configs,
# plan, and relevant scripts so inference never needs a separate manual upload.
runtime_dirs = list(DATA_ROOT.rglob("runtime_bundle")) if DATA_ROOT.is_dir() else []
artifact_paths.extend(runtime_dirs)
if DATASET_AUDIT is not None and audit_candidates:
    artifact_paths.append(audit_candidates[0])
if INCLUDE_FULL_DATASET_IN_ARTIFACT_ZIP:
    artifact_paths.append(RESULTS_CSV)

ARTIFACT_ZIP = package_paths(
    RESULTS_DIR / f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}_complete_artifacts.zip",
    artifact_paths, root=WORKSPACE,
)
print("Artifact ZIP:", ARTIFACT_ZIP)
print("Size MB:", ARTIFACT_ZIP.stat().st_size / 1024**2)
print("SHA-256:", sha256_file(ARTIFACT_ZIP))
if run is not None:
    artifact = wandb_module.Artifact(
        name=f"{RUN_NAME}_{SELECTED_CHECKPOINT_ROLE}", type="model-run",
        metadata={"model_id": MODEL_ID, "checkpoint_role": SELECTED_CHECKPOINT_ROLE},
    )
    artifact.add_file(str(ARTIFACT_ZIP))
    run.log_artifact(artifact)
    run.finish()
download_if_colab(ARTIFACT_ZIP, enabled=AUTO_DOWNLOAD_FINAL_ZIP)
